# builder — сборка и выпуск базы подсказок

Тетрадка заменяет `build.bat` + `precompute.py` и `publish.bat` + `publish.py`.
Venv и `run_app.bat` не нужны: всё работает в ядре Jupyter, недостающие пакеты ставятся в шаге 1.

**Обычный цикл**
1. Положить новые JSON в `builder\data\checks\`.
2. *Run All* (выпуск по умолчанию выключен) или ячейки шагов 0–6 — база запишется в `..\base\hints.db`.
   Проверить результат: `..\start.bat` (приложение сразу видит новую базу).
3. Раздел «Выпуск»: поставить `RELEASE = True` (и при желании `EXTRA_CHANGES`, `PUSH = True`), выполнить шаги 7–8 — поднимется VERSION,
   допишется CHANGELOG.md, по желанию будут сделаны `git commit` и `push`.

Тетрадка должна лежать в папке `builder\` (рядом с `data\checks\`).

## 0. Параметры сборки
То, что раньше передавалось аргументами `build.bat --llm … --emb … --dedup … --src … --out …`.

In [ ]:
from pathlib import Path

LLM = None      # модель Ollama для тегов и объяснений, напр. "qwen2.5:7b"; None — словарь TAXONOMY
EMB = None      # модель sentence-transformers, напр. "intfloat/multilingual-e5-base"; None — TF-IDF
DEDUP = None    # порог объединения одинаковых задач; None — 0.85 для EMB, 0.35 для TF-IDF

# Папка builder\: тетрадка лежит в ней, Jupyter запускает ядро в папке тетрадки.
BASE = Path.cwd()
if not (BASE / "data" / "checks").exists() and (BASE / "builder" / "data" / "checks").exists():
    BASE = BASE / "builder"
APP_DIR = BASE.parent                     # папка приложения пользователей

SRC_DIR = BASE / "data" / "checks"        # JSON проверок (раньше --src)
DB = APP_DIR / "base" / "hints.db"        # файл базы (раньше --out)
SCHEMA_VERSION = 1                        # формат базы; приложение проверяет его при открытии

print("builder:", BASE)
print("JSON:   ", SRC_DIR, "(есть)" if SRC_DIR.exists() else "(НЕТ ПАПКИ!)")
print("база:   ", DB)

## 1. Зависимости
То, что делал `run_app.bat`: ставит недостающие пакеты (в окружение текущего ядра).

In [ ]:
import importlib.util, subprocess, sys

need = {"numpy": "numpy", "sklearn": "scikit-learn"}
if EMB:
    need["sentence_transformers"] = "sentence-transformers"
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Ставлю:", *missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()
print("Python:", sys.version.split()[0], "| зависимости на месте")

## 2. Настройки методики
Таксономия процедур, веса, пороги — как в начале `precompute.py`.

In [ ]:
# Таксономия процедур («суть» проверки). Ключи — регулярные выражения по нижнему регистру.
TAXONOMY = {
    "сравнение цен с рынком": [r"рыночн", r"внешн(ей|ие|их) (сет|источник)", r"интернет", r"маркетплейс", r"открытых источник", r"завыш"],
    "конкурентная процедура": [r"конкурент", r"коммерческ\w* предлож", r"выбор\w* поставщик", r"торг", r"тендер", r"закупочн"],
    "соответствие спецификации": [r"спецификац", r"характеристик", r"сопоставим"],
    "соответствие нормативным требованиям": [r"сертификат", r"декларац\w* соответств", r"-фз", r"техрегламент", r"тр тс", r"пожарн", r"законодательств"],
    "контроль качества": [r"входн\w* контрол", r"качеств", r"приёмк|приемк", r"дефект"],
    "аффилированность": [r"аффилир", r"конфликт\w* интерес", r"связ\w* .*учредител"],
    "полномочия подписанта": [r"полномоч", r"подписант", r"доверенност"],
    "соблюдение сроков": [r"срок", r"просроч", r"исполнительск"],
    "выборочное тестирование": [r"выборк", r"репрезентатив"],
    "подтверждение исполнения": [r"\bакт(ы|ов|ами|ах)\b", r"фактическ\w* оказ", r"объём\w* услуг|объем\w* услуг", r"смет"],
    "ответственность и организация": [r"виновн", r"ответственн", r"проектн\w* команд", r"назначен"],
}

SIM_TOP_K = 8            # сколько похожих проверок брать
SIM_MIN = 0.15           # минимальная похожесть
W_TAGS, W_TEXT = 0.6, 0.4
DEDUP_TFIDF, DEDUP_EMB = 0.35, 0.85   # порог объединения одинаковых задач
CONFIRMED_BOOST = 1.5    # задача, по которой нарушение подтвердилось, весит больше
TYPICAL_SHARE = 0.5      # задача есть в ≥50% открытых проверок → «типовая»
TYPICAL_MIN = 3
# Задачи, которые всегда считаются типовыми (оформление, а не суть). Регулярные выражения.
TYPICAL_PATTERNS = [r"написан\w* акта", r"хронолог"]

dedup = DEDUP or (DEDUP_EMB if EMB else DEDUP_TFIDF)
print("порог объединения задач:", dedup)

## 3. Функции

In [ ]:
import json, os, re, sqlite3, time
import numpy as np


def clean(s):
    return re.sub(r"\s+", " ", str(s or "")).strip()


# ---------------- загрузка ----------------

def load_checks(src_dir):
    raw = []
    for f in sorted(Path(src_dir).glob("*.json")):
        d = json.loads(f.read_text(encoding="utf-8"))
        raw.extend(d if isinstance(d, list) else [d])
    checks = []
    for r in raw:
        c = {
            "id": str(r.get("id")),
            "number": r.get("number"),
            "title": clean(r.get("name")) or f"Проверка {r.get('number')}",
            "purpose": clean(r.get("program_tasks")),
            "metrics": clean(r.get("program_metics") or r.get("program_metrics")),
            "background": clean(r.get("program_points")),
            "items": [],   # задачи (или гипотезы без задач)
        }
        for h in r.get("hypos") or []:
            ht = clean(h.get("hypo_title"))
            tasks = h.get("hypo_tasks") or []
            if not tasks and ht:
                tasks = [{"task_title": ht, "task_confirmed": h.get("hypo_confirmed"),
                          "task_results": h.get("hypo_results")}]
            for t in tasks:
                tt = clean(t.get("task_title"))
                if not tt:
                    continue
                conf = t.get("task_confirmed")
                c["items"].append({
                    "task": tt, "hypo": ht,
                    "confirmed": conf if conf is not None else h.get("hypo_confirmed"),
                    "results": clean(t.get("task_results")),
                    "invoice": clean(t.get("task_invoice")),
                    "method": clean(t.get("task_method")),
                })
        c["opened"] = bool(c["items"])
        hyp = list(dict.fromkeys(i["hypo"] for i in c["items"] if i["hypo"]))
        c["text"] = ". ".join([c["title"], c["purpose"], c["metrics"], c["background"], *hyp,
                               *(i["task"] for i in c["items"])])
        c["short"] = ". ".join([c["title"], c["purpose"], c["metrics"]])
        checks.append(c)
    return checks


# ---------------- теги и объяснения ----------------

def tags_by_keywords(text):
    t = text.lower()
    return sorted(tag for tag, kws in TAXONOMY.items() if any(re.search(k, t) for k in kws))


def ollama(model, prompt, as_json=False):
    import urllib.request
    body = {"model": model, "prompt": prompt, "stream": False}
    if as_json:
        body["format"] = "json"
    req = urllib.request.Request("http://localhost:11434/api/generate",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())["response"]


def tags_by_llm(text, model):
    prompt = ("Ты помощник аудитора. Определи методические процедуры проверки (суть, а не предмет). "
              "Выбери теги ТОЛЬКО из списка: " + "; ".join(TAXONOMY) +
              '. Ответь JSON вида {"tags": [...]}.\n\nПроверка:\n' + text[:4000])
    try:
        d = json.loads(ollama(model, prompt, True))
        d = d.get("tags", d) if isinstance(d, dict) else d
        tags = [x for x in d if x in TAXONOMY]
    except Exception:
        tags = []
    return sorted(set(tags) or tags_by_keywords(text))


def reason_by_tags(c, o):
    common = sorted(set(c["tags"]) & set(o["tags"]))
    only_o = sorted(set(o["tags"]) - set(c["tags"]))
    only_c = sorted(set(c["tags"]) - set(o["tags"]))
    parts = ["Общее: " + (", ".join(common) if common else "близкие по тексту описания")]
    if only_o:
        parts.append("только в источнике: " + ", ".join(only_o))
    if only_c:
        parts.append("только в этой проверке: " + ", ".join(only_c))
    return "; ".join(parts) + "."


def reason_by_llm(c, o, model):
    prompt = ("Ты помощник аудитора. Кратко (1–2 предложения, по-русски) объясни, чем по сути (методике) "
              "похожи две проверки и какое различие важно при переносе задач из второй в первую. Без вступлений.\n\n"
              f"Проверка 1: {c['text'][:1500]}\n\nПроверка 2: {o['text'][:1500]}")
    try:
        return ollama(model, prompt).strip()
    except Exception:
        return reason_by_tags(c, o)


# ---------------- векторы ----------------

_st_model = None   # модель sentence-transformers грузится один раз за сессию

def embed(texts, emb_model):
    global _st_model
    if not texts:
        return np.zeros((0, 1), dtype=np.float32)
    if emb_model:
        from sentence_transformers import SentenceTransformer
        if _st_model is None or _st_model[0] != emb_model:
            _st_model = (emb_model, SentenceTransformer(emb_model))
        return np.asarray(_st_model[1].encode(texts, normalize_embeddings=True), dtype=np.float32)
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit_transform(texts)
    return normalize(X).astype(np.float32)  # разрежённая матрица


def dot(A, B):
    r = A @ B.T
    return r.toarray() if hasattr(r, "toarray") else np.asarray(r)


def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else 0.0

## 4. Загрузка проверок и теги

In [ ]:
checks = load_checks(SRC_DIR)
if not checks:
    raise RuntimeError(f"Нет JSON проверок в {SRC_DIR}")
print(f"Проверок: {len(checks)}, из них без задач: {sum(not c['opened'] for c in checks)}")

for c in checks:
    c["tags"] = tags_by_llm(c["text"], LLM) if LLM else tags_by_keywords(c["text"])
    print(f"  {c['number']}: {c['tags']}")

## 5. Сходство, похожие проверки и подсказки
Всё считается в памяти — до записи в базу можно посмотреть результат (ячейка ниже).

In [ ]:
# Для неоткрытых проверок есть только описание — тогда сравниваем «короткие» описания.
E_full = embed([c["text"] for c in checks], EMB)
E_short = embed([c["short"] for c in checks], EMB)
sim_full, sim_short = dot(E_full, E_full), dot(E_short, E_short)

all_items = [(ci, it) for ci, c in enumerate(checks) for it in c["items"]]
QE = embed([it["task"] for _, it in all_items], EMB)
QS = dot(QE, QE) if all_items else np.zeros((0, 0))

# «Типовые» задачи: похожая задача встречается в большой доле проверок
opened = max(1, sum(c["opened"] for c in checks))
typical = []
for k in range(len(all_items)):
    cnt = len({all_items[m][0] for m in np.where(QS[k] >= dedup)[0]})
    by_pattern = any(re.search(p, all_items[k][1]["task"].lower()) for p in TYPICAL_PATTERNS)
    typical.append(by_pattern or (cnt >= TYPICAL_MIN and cnt / opened >= TYPICAL_SHARE))

similar_rows, hint_rows = [], []   # строки таблиц similar и hints
for i, c in enumerate(checks):
    sims = []
    for j, o in enumerate(checks):
        if i == j or not o["opened"]:
            continue  # неоткрытая проверка не может быть источником задач
        ts = sim_full[i, j] if c["opened"] else sim_short[i, j]
        s = W_TAGS * jaccard(c["tags"], o["tags"]) + W_TEXT * float(ts)
        if s >= SIM_MIN:
            sims.append((s, j))
    sims = sorted(sims, reverse=True)[:SIM_TOP_K]
    weight = {j: s for s, j in sims}
    for s, j in sims:
        o = checks[j]
        common = sorted(set(c["tags"]) & set(o["tags"]))
        reason = reason_by_llm(c, o, LLM) if LLM else reason_by_tags(c, o)
        similar_rows.append((c["id"], o["id"], round(s, 3), json.dumps(common, ensure_ascii=False), reason))

    # Задачи похожих проверок → объединяем дубли
    idx = [k for k, (ci, _) in enumerate(all_items) if ci in weight]
    clusters = []
    for k in sorted(idx, key=lambda k: -weight[all_items[k][0]]):
        for cl in clusters:
            if QS[k, cl[0]] >= dedup:
                cl.append(k)
                break
        else:
            clusters.append([k])

    own = [k for k, (ci, _) in enumerate(all_items) if ci == i]  # задачи самой проверки
    hints = []
    for cl in clusters:
        src = {}
        for k in cl:
            ci, it = all_items[k]
            o = checks[ci]
            s = src.setdefault(o["id"], {"check_id": o["id"], "number": o["number"],
                                         "score": round(weight[ci], 3), "items": []})
            s["items"].append(it)
        conf = sum(1 for k in cl if all_items[k][1]["confirmed"])
        score = sum(weight[all_items[k][0]] * (CONFIRMED_BOOST if all_items[k][1]["confirmed"] else 1)
                    for k in cl)
        rep = all_items[cl[0]][1]
        # есть ли похожая задача уже в программе этой проверки
        covered = ""
        if own:
            best = max(own, key=lambda m: QS[cl[0], m])
            if QS[cl[0], best] >= dedup:
                covered = all_items[best][1]["task"]
        hints.append((int(any(typical[k] for k in cl)), -score, rep["task"], rep["hypo"],
                      conf, len(cl), covered, list(src.values())))
    hints.sort(key=lambda h: (h[0], h[1]))  # сначала содержательные, типовые в конце
    for no, (typ, nscore, task, hypo, conf, total, covered, src) in enumerate(hints, 1):
        hint_rows.append((c["id"], no, task, hypo, round(-nscore, 3), typ, conf, total, covered,
                          json.dumps(src, ensure_ascii=False)))

print(f"Задач всего: {len(all_items)}, типовых: {sum(typical)}")
print(f"Связей «похожая проверка»: {len(similar_rows)}, подсказок: {len(hint_rows)}")

### Просмотр (необязательно)
Укажите номер проверки — покажет похожие проверки и первые подсказки так, как их увидит приложение.

In [ ]:
SHOW_NUMBER = checks[0]["number"]   # номер проверки для просмотра
TOP_HINTS = 10

c = next(x for x in checks if str(x["number"]) == str(SHOW_NUMBER))
num = {x["id"]: x["number"] for x in checks}
print(f"{c['number']} — {c['title']}\nтеги: {c['tags']}\n\nПохожие:")
for cid, sid, score, _, reason in (r for r in similar_rows if r[0] == c["id"]):
    print(f"  {num[sid]}  {score:.3f}  {reason}")
print("\nПодсказки:")
for r in [r for r in hint_rows if r[0] == c["id"]][:TOP_HINTS]:
    _, no, task, hypo, score, typ, conf, total, covered, _ = r
    flags = (" [типовая]" if typ else "") + (f" [подтв. {conf}/{total}]" if conf else "") + (" [есть в программе]" if covered else "")
    print(f"  {no:>2}. {score:.3f}  {task}{flags}")

## 6. Запись базы
Пишем во временный файл и подменяем в конце: запущенное приложение всё это время видит прежнюю базу.

In [ ]:
DB.parent.mkdir(parents=True, exist_ok=True)
tmp_db = DB.with_suffix(".db.tmp")
tmp_db.unlink(missing_ok=True)
db = sqlite3.connect(tmp_db)
try:
    db.executescript("""
        CREATE TABLE meta(key TEXT PRIMARY KEY, value TEXT);
        CREATE TABLE checks(id TEXT PRIMARY KEY, number TEXT, title TEXT, purpose TEXT, metrics TEXT,
                            background TEXT, tags TEXT, opened INTEGER, items TEXT);
        CREATE TABLE similar(check_id TEXT, similar_id TEXT, score REAL, common_tags TEXT, reason TEXT);
        CREATE TABLE hints(check_id TEXT, hint_no INTEGER, task TEXT, hypo TEXT, score REAL,
                           typical INTEGER, confirmed INTEGER, total INTEGER, covered TEXT, sources TEXT);
    """)
    db.executemany("INSERT INTO checks VALUES(?,?,?,?,?,?,?,?,?)",
                   [(c["id"], str(c["number"]), c["title"], c["purpose"], c["metrics"], c["background"],
                     json.dumps(c["tags"], ensure_ascii=False), int(c["opened"]),
                     json.dumps(c["items"], ensure_ascii=False)) for c in checks])
    db.executemany("INSERT INTO similar VALUES(?,?,?,?,?)", similar_rows)
    db.executemany("INSERT INTO hints VALUES(?,?,?,?,?,?,?,?,?,?)", hint_rows)

    meta = {"schema_version": SCHEMA_VERSION, "built_at": time.strftime("%Y-%m-%d %H:%M"),
            "checks": len(checks), "opened": sum(c["opened"] for c in checks),
            "llm": LLM or "", "emb": EMB or "", "dedup": dedup}
    db.executemany("INSERT INTO meta VALUES(?,?)", [(k, str(v)) for k, v in meta.items()])
    db.commit()
finally:
    db.close()
os.replace(tmp_db, DB)
print(f"Готово: {DB}")
print(f"База от {meta['built_at']}: {meta['checks']} проверок ({meta['opened']} с задачами)")

---
# Выпуск новой базы для пользователей
То, что делал `publish.bat`: поднять VERSION, дописать CHANGELOG.md и (по желанию) `git commit` + `push`.
Пользователи получат базу кнопкой «Обновить и перезапустить».

Если база уже собрана раньше (бывший `publish.bat --no-build`), шаги 0–6 можно не выполнять — достаточно шага 0 и этих ячеек.

## 7. Новая версия

In [ ]:
RELEASE = False         # True — выпустить версию (защита от случайного Run All)
EXTRA_CHANGES = []      # что ещё изменилось, пункты для CHANGELOG, напр. ["Исправлен поиск"]
BUMP = "patch"          # "patch" 1.0.015 → 1.0.016 | "minor" → 1.1.001 | "major" → 2.0.001
DRY_RUN = False         # True — только показать, файлы не трогать

import sqlite3, subprocess, sys, os
from pathlib import Path

released = False
if not RELEASE:
    print("RELEASE = False — версия не поднималась. Поставьте True и выполните ячейку ещё раз.")
elif not DB.exists():
    raise FileNotFoundError(f"Нет базы {DB} — сначала выполните шаги 0–6")
else:
    with sqlite3.connect(f"file:{DB}?mode=ro", uri=True) as con:
        db_meta = dict(con.execute("SELECT key, value FROM meta"))
    msg = f"Обновлена база подсказок: {db_meta.get('checks')} проверок, сборка {db_meta.get('built_at')}"
    print("Пункт истории:", msg)

    cmd = [sys.executable, str(APP_DIR / "release" / "bump_version.py"), "-c", msg]
    for extra in EXTRA_CHANGES:
        if extra.strip():
            cmd += ["-c", extra.strip()]
    if BUMP in ("minor", "major"):
        cmd.append(f"--{BUMP}")
    if DRY_RUN:
        cmd.append("--dry-run")

    r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace",
                       env={**os.environ, "PYTHONUTF8": "1"}, stdin=subprocess.DEVNULL)
    print(r.stdout, r.stderr, sep="")
    if r.returncode != 0:
        raise RuntimeError("bump_version.py завершился с ошибкой — версия не поднята")
    version = (APP_DIR / "VERSION").read_text(encoding="utf-8").strip()
    print("VERSION сейчас:", version)
    released = not DRY_RUN

## 8. Commit и push
По умолчанию выключено — поставьте `PUSH = True`, когда проверили базу и версию.

In [ ]:
PUSH = False

import shutil
git = shutil.which("git")
manual = 'git add base/hints.db VERSION CHANGELOG.md && git commit -m "Новая база" && git push'

def run_git(*args):
    print(">", "git", *args)
    r = subprocess.run([git, "-C", str(APP_DIR), *args], capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    print(r.stdout, r.stderr, sep="")
    if r.returncode != 0:
        raise RuntimeError("git завершился ошибкой — проверьте вывод выше")

if not git or not (APP_DIR / ".git").exists():
    print("git или репозиторий не найден. Закоммитьте и отправьте в Bitbucket вручную:\n ", manual)
elif not globals().get("released"):
    print("Сначала выпустите версию в шаге 7 (RELEASE = True).")
elif not PUSH:
    print(f"PUSH = False — версия {version} не отправлена. Вручную:\n ", manual)
else:
    run_git("add", "base/hints.db", "VERSION", "CHANGELOG.md")
    run_git("commit", "-m", f"Версия {version}: {msg}")
    run_git("push")
    print(f"Готово: версия {version} в Bitbucket. Пользователи увидят обновление при следующем открытии.")